# NeurALPS · interactive assembly explorer

Inspect domain cartoons on the original natural-reference scale, score any selection jointly, and rank matching natural fragments in your assembly.

- **A**: circle · **T/PCP**: small square · **C**: diamond · **TE**: hexagon. Boundaries are clickable.
- **Natural-reference percentile** uses the original, type-matched full-context single-object scale from Jean Zay.
- **Natural-repertoire rank** compares candidates against the same jointly masked query. These two numbers answer different questions.
- **Supervised activity** is the model score from the head trained on all 494 reviewed assemblies; it is not a calibrated activity probability.

Use a fresh runtime with an A100 or L4 for exact BF16 ESMC extraction. No source or runtime ZIP uploads are required.


## 1 · Load the published source
The resolved Git commit is printed and recorded. Change REVISION to a full commit hash to reproduce an older run.


In [ ]:
REPO_URL = "https://github.com/AMIRMOHAMMAD-OSS/NeurALPS-v2" #@param {type:"string"}
REVISION = "main" #@param {type:"string"}
from pathlib import Path
import subprocess, sys, json, hashlib, re
BASE = Path("/content/neuralps_work")
BASE.mkdir(exist_ok=True)
ROOT = BASE / "repository"
if not REPO_URL.startswith("https://github.com/") or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", REVISION):
    raise ValueError("Provide a GitHub URL and valid revision")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "origin", REVISION], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT), "matplotlib", "huggingface-hub>=0.34,<1"], check=True)
sys.path.insert(0, str(ROOT))
print("Source commit:", SOURCE_COMMIT)


## 2 · Inspect the fitted supervised model
This demonstration uses saved training features. These predictions are **in sample**, and do not measure generalization.

In [ ]:
import numpy as np
from scipy.special import expit
from neuralps.head import load_model, predict
from neuralps.contracts import sha
HEAD = ROOT/"models/all494"
head_manifest = json.loads((HEAD/"manifest.json").read_text())
assert sha(HEAD/"supervised_head.npz") == head_manifest["model_sha256"]
print("Training rows:", head_manifest["trained_rows"], head_manifest["by_dataset"])
print("L2 penalty:", head_manifest["l2"], "— encoder updates:", head_manifest["encoder_updates"])
rows = json.loads((ROOT/"data/training_rows.json").read_text())
with np.load(ROOT/"data/training_features.npz", allow_pickle=False) as z:
    TRAIN_X = z["features"]
head = load_model(HEAD/"supervised_head.npz")
example_index = 0 #@param {type:"integer"}
if not 0 <= example_index < len(rows):
    raise ValueError("example_index must be 0–493")
print(rows[example_index])
print("In-sample supervised activity score:", float(expit(predict(head, TRAIN_X[example_index:example_index+1])[0])))


## 3 · Download runtime assets and verify encoder parity
The published runtime bundle is downloaded automatically, checked against its pinned SHA-256, and tested against original-code golden fixtures.


In [ ]:
from neuralps.downloads import download, extract
RUNTIME_URL = "https://github.com/AMIRMOHAMMAD-OSS/NeurALPS-v2/releases/download/v0.1.0-20261009/NeurALPS_runtime_20261009.zip"
RUNTIME_SHA256 = "23ea8591e52a63f23d5ff8d22e01e58a60d75c0a9ffd1bc9622e167d9de6528f"
archive = download(RUNTIME_URL, BASE / "runtime.zip", RUNTIME_SHA256)
ASSETS = BASE / "runtime_assets"
if not ASSETS.exists():
    extract(archive, ASSETS)
manifest = json.loads((ASSETS / "manifest.json").read_text())
for name, digest in manifest["files_sha256"].items():
    target = (ASSETS / name).resolve()
    if not target.is_relative_to(ASSETS.resolve()) or sha(target) != digest:
        raise ValueError("Runtime checksum mismatch: " + name)
subprocess.run([sys.executable, str(ROOT / "scripts/verify_runtime.py"), "--assets", str(ASSETS), "--report", str(BASE / "encoder_parity.json")], check=True)


## 3b · Load the original natural-reference scale
These assets require the one-time Jean Zay export in `docs/EXPLORER_SETUP.md`. The notebook never substitutes a fabricated percentile when the reference is absent.


In [ ]:
NATURAL_RELEASE_URL = "https://github.com/AMIRMOHAMMAD-OSS/NeurALPS-v2/releases/download/v0.2.0-20261010" #@param {type:"string"}
from neuralps.downloads import natural_reference, natural_repertoire
import urllib.error
try:
    reference = natural_reference(BASE, NATURAL_RELEASE_URL)
    print("Original natural-reference scale loaded and verified.")
except urllib.error.HTTPError as e:
    if e.code != 404:
        raise
    reference = None
    print("Natural-reference asset is not yet published. Scores will remain unscaled.")
    print("Run the one-time export and release upload in docs/EXPLORER_SETUP.md, then rerun this cell and the explorer cell.")


## 4 · Install the preserved ESMC implementation
The installed fork is exported with its metadata and licenses because its recorded Git URL was unavailable during porting. Every Python source file is checked before inference.

In [ ]:
if "transformers" in sys.modules:
    raise RuntimeError("Restart the runtime before installing the preserved Transformers fork, then rerun setup")
wheel = ASSETS/manifest["transformers_distribution"]["wheel"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.22.1", "safetensors>=0.4.3", "filelock", "packaging", "pyyaml", "regex", "requests", "tqdm", "huggingface-hub>=0.34,<1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", str(wheel)], check=True)
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Select a BF16-capable GPU runtime for exact ESMC extraction. The encoder checks above work on CPU.")
print(torch.cuda.get_device_name(0))


## 5 · Obtain the pinned ESMC weights
The download uses the original model revision and verifies the weight SHA-256. If you already have the same model folder on mounted storage, enter its path.

In [ ]:
LOCAL_ESMC_DIRECTORY = "" #@param {type:"string"}
from neuralps.embedding import download_esmc
ESMC = Path(LOCAL_ESMC_DIRECTORY) if LOCAL_ESMC_DIRECTORY else download_esmc(BASE/"ESMC-600M")
print("Pinned model folder:", ESMC)


## 6 · Choose an annotated assembly
Keep the included example for your first test, or upload an annotated assembly JSON. Domain coordinates are 0-based and end-exclusive. Segment selection is available inside the explorer.


In [ ]:
INPUT = "example" #@param ["example", "upload JSON"]
from google.colab import files
if INPUT == "example":
    spec = json.loads((ROOT / "examples/AI_1_annotated.json").read_text())
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one annotated assembly JSON")
    spec = json.loads(next(iter(uploaded.values())).decode("utf-8"))
print("Assembly:", spec["assembly_id"])


## 7 · Open the interactive explorer
Click a part for its percentile and coordinates. Shift-click selects a contiguous segment. “Find natural alternatives” downloads the natural-repertoire index and exact cached embeddings as needed. “Test variant” rebuilds and re-embeds a complete-domain splice within one protein.


In [ ]:
OUTPUTS = "both" #@param ["both", "pretrained only"]
from neuralps.explorer import ExplorerSession, show_explorer
session = ExplorerSession.from_spec(
    spec, ASSETS, ESMC,
    head_dir=HEAD if OUTPUTS == "both" else None,
    reference=reference,
    repertoire_factory=lambda: natural_repertoire(BASE, NATURAL_RELEASE_URL),
)
session.result["source_commit"] = SOURCE_COMMIT
result = session.result
show_explorer(session)


**Reading the explorer**

- A natural-reference score of 80 means this object ranks at approximately the 80th percentile among matched natural reference objects in their original contexts. Reference groups need at least 100 independent connected components under the original protocol. Unsupported scales remain gray.
- A segment's natural-repertoire rank answers how it compares with same-type, same-topology natural fragments in this masked assembly. Exact sequence duplicates are counted once; alternatives visible elsewhere in the unmasked context are excluded.
- Natural donor scores are retrieval scores. “Test variant” rebuilds recipient boundary windows and reruns ESMC plus NeurALPS; the tested activity-head score is still uncalibrated. Boundary-only and cross-protein edits can be exported as FASTA, but are not automatically spliced.
- Saved HTML retains inspection and already-computed results. New model queries require the live Colab runtime.


## 8 · Save the interactive result
The HTML is self-contained. It preserves your computed segment scores and candidate tables. Use the live notebook for new computations.


In [ ]:
from neuralps.explorer import save_explorer
from neuralps.contracts import dump
html_path = save_explorer(session, BASE / "neuralps_explorer.html")
result_path = BASE / "neuralps_explorer.json"
dump(result_path, session.payload())
files.download(str(html_path))
files.download(str(result_path))


## Optional · Reproduce the all-label fit
The saved head is already trained. This refits the same 494 rows and checks exact membership before optimization. The output directory must be new. It does not create a new held-out evaluation.

In [ ]:
REFIT = False #@param {type:"boolean"}
if REFIT:
    subprocess.run([sys.executable, str(ROOT/"scripts/fit_all_labels.py"), "--output", str(BASE/"refit_all494")], check=True)
